In [ ]:
!pip install sentence-transformers pandas scikit-learn

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [ ]:
df = pd.read_csv('clickbait_data.csv')
print(df.columns) # This will print the exact column names

Index(['headline', 'clickbait'], dtype='object')


In [ ]:
# 1. Load your dataset
df = pd.read_csv('clickbait_data.csv')

# UPDATE: Using your exact column names here
headlines = df['headline'].tolist()
labels = df['clickbait'].tolist()

# 2. Initialize the Sentence Transformer model
print("Loading Transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# 3. Generate Embeddings
print("Encoding headlines into semantic vectors...")
embeddings = model.encode(headlines, show_progress_bar=True)

print(f"Shape of embeddings: {embeddings.shape}")

Loading Transformer model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding headlines into semantic vectors...


Batches:   0%|          | 0/1000 [00:00<?, ?it/s]

Shape of embeddings: (32000, 384)


In [ ]:
# 1. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(embeddings, labels, test_size=0.2, random_state=42)

# 2. Initialize and train the SVM Classifier
print("Training the SVM Classifier on semantic embeddings...")
clf = SVC(kernel='linear', C=1.0, probability=True) # Added probability=True for deeper analysis
clf.fit(X_train, y_train)
print("Training complete!\n")

# 3. Make predictions on the test set
y_pred = clf.predict(X_test)

# 4. Generate Performance Metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"Semantic View Accuracy: {accuracy * 100:.2f}%\n")

print("--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Non-Clickbait', 'Clickbait']))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

Training the SVM Classifier on semantic embeddings...
Training complete!

Semantic View Accuracy: 96.58%

--- Detailed Classification Report ---
               precision    recall  f1-score   support

Non-Clickbait       0.96      0.97      0.97      3127
    Clickbait       0.97      0.97      0.97      3273

     accuracy                           0.97      6400
    macro avg       0.97      0.97      0.97      6400
 weighted avg       0.97      0.97      0.97      6400

--- Confusion Matrix ---
[[3022  105]
 [ 114 3159]]


In [ ]:
# Create a DataFrame of the test set results to find exactly where the model failed
test_indices = np.arange(len(labels))[len(y_train):] # Approximation for alignment, better to use X_test indices if kept
# To be safe and exact with the test text:
X_train_text, X_test_text, y_train_labels, y_test_labels = train_test_split(headlines, labels, test_size=0.2, random_state=42)

# Get confidence scores
probs = clf.predict_proba(X_test)[:, 1]

# Build a results dataframe
results_df = pd.DataFrame({
    'Headline': X_test_text,
    'True_Label': y_test_labels,
    'Predicted_Label': y_pred,
    'Clickbait_Probability': probs
})

# Find False Positives (Predicted Clickbait, but was actually Non-Clickbait)
# Sorted by how confident the model was in its wrong answer
false_positives = results_df[(results_df['True_Label'] == 0) & (results_df['Predicted_Label'] == 1)]
false_positives = false_positives.sort_values(by='Clickbait_Probability', ascending=False)

print("--- Top 3 False Positives (Model wrongly thought these were clickbait) ---")
for idx, row in false_positives.head(3).iterrows():
    print(f"Prob: {row['Clickbait_Probability']:.2f} | {row['Headline']}")

print("\n")

# Find False Negatives (Predicted Non-Clickbait, but was actually Clickbait)
false_negatives = results_df[(results_df['True_Label'] == 1) & (results_df['Predicted_Label'] == 0)]
false_negatives = false_negatives.sort_values(by='Clickbait_Probability', ascending=True)

print("--- Top 3 False Negatives (Clickbait that slipped past the model) ---")
for idx, row in false_negatives.head(3).iterrows():
    print(f"Prob: {row['Clickbait_Probability']:.2f} | {row['Headline']}")

--- Top 3 False Positives (Model wrongly thought these were clickbait) ---
Prob: 1.00 | Remembering the Way It Was
Prob: 1.00 | An Australian child's vocabulary: it's "I" before "we", both before "you"
Prob: 1.00 | Male Mice Sing to Woo, but the Females Answer Just a First Call


--- Top 3 False Negatives (Clickbait that slipped past the model) ---
Prob: 0.00 | A Dutch Organization Is Providing Free Abortions For Women In Zika-Affected Regions
Prob: 0.00 | Hark, These Public Libraries Are Dueling To The Death to Defend Their Baseball Teams
Prob: 0.00 | Controversial Boxer Tyson Fury Stripped Of Heavyweight Title Just Two Weeks After Winning It


In [ ]:
import pandas as pd
import numpy as np
import time
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. Load the Dataset
print("Loading dataset...")
df = pd.read_csv('clickbait_data.csv')
headlines = df['headline'].tolist()
labels = df['clickbait'].tolist()

# 2. Generate Embeddings (Only needs to happen once)
print("Loading Transformer model (all-MiniLM-L6-v2)...")
transformer = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding headlines into semantic vectors...")
embeddings = transformer.encode(headlines, show_progress_bar=True)
print(f"Embeddings generated with shape: {embeddings.shape}\n")

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(embeddings, labels, test_size=0.2, random_state=42)

# 4. Define the Models
# We initialize all three models with standard, robust hyperparameters
models = {
    "Support Vector Machine": SVC(kernel='linear', C=1.0, probability=True, random_state=42),
    "Multi-Layer Perceptron": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, activation='relu', random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
}

# Dictionary to store the final results for our comparison table
results = []

# 5. Train and Evaluate Each Model
print("--- Starting Model Training & Evaluation ---\n")

for name, model in models.items():
    print(f"Training {name}...")
    start_time = time.time()

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate execution time
    execution_time = time.time() - start_time

    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro') # Macro averages both clickbait and non-clickbait performance

    # Save to results
    results.append({
        "Model": name,
        "Accuracy": acc * 100,
        "F1-Score": f1 * 100,
        "Training Time (s)": execution_time
    })

    # Print individual detailed report
    print(f"✅ {name} Complete!")
    print(classification_report(y_test, y_pred, target_names=['Non-Clickbait', 'Clickbait']))
    print("-" * 50 + "\n")

# 6. Final Comparative Analysis Table
print("=== FINAL BENCHMARK COMPARISON ===")
results_df = pd.DataFrame(results)
results_df['Accuracy'] = results_df['Accuracy'].round(2).astype(str) + '%'
results_df['F1-Score'] = results_df['F1-Score'].round(2).astype(str) + '%'
results_df['Training Time (s)'] = results_df['Training Time (s)'].round(3)

# Display the clean table
print(results_df.to_markdown(index=False))

Loading dataset...
Loading Transformer model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding headlines into semantic vectors...


Batches:   0%|          | 0/1000 [00:00<?, ?it/s]

Embeddings generated with shape: (32000, 384)

--- Starting Model Training & Evaluation ---

Training Support Vector Machine...
✅ Support Vector Machine Complete!
               precision    recall  f1-score   support

Non-Clickbait       0.96      0.97      0.97      3127
    Clickbait       0.97      0.97      0.97      3273

     accuracy                           0.97      6400
    macro avg       0.97      0.97      0.97      6400
 weighted avg       0.97      0.97      0.97      6400

--------------------------------------------------

Training Multi-Layer Perceptron...
✅ Multi-Layer Perceptron Complete!
               precision    recall  f1-score   support

Non-Clickbait       0.97      0.98      0.97      3127
    Clickbait       0.98      0.97      0.98      3273

     accuracy                           0.98      6400
    macro avg       0.98      0.98      0.98      6400
 weighted avg       0.98      0.98      0.98      6400

-------------------------------------------------

In [ ]:
import pandas as pd
import time
from sentence_transformers import SentenceTransformer
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. Helper function to load and merge specific jsonl files
def load_clickbait_data(instances_file, truth_file):
    print(f"Loading {instances_file} and {truth_file}...")

    # Read the JSONL files
    instances_df = pd.read_json(instances_file, lines=True)
    truth_df = pd.read_json(truth_file, lines=True)

    # Merge on the shared 'id'
    merged_df = pd.merge(instances_df, truth_df, on='id')

    # Extract text: Webis dataset uses 'postText' (usually a list) or 'targetTitle'
    if 'postText' in merged_df.columns:
        headlines = merged_df['postText'].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x)).tolist()
    elif 'targetTitle' in merged_df.columns:
        headlines = merged_df['targetTitle'].tolist()
    else:
        headlines = merged_df['headline'].tolist()

    # Convert 'clickbait' / 'no-clickbait' string labels to 1 and 0
    if merged_df['truthClass'].dtype == object:
        labels = (merged_df['truthClass'] == 'clickbait').astype(int).tolist()
    else:
        labels = merged_df['truthClass'].tolist()

    return headlines, labels

# 2. Load Train, Validation, and Test datasets directly
train_headlines, y_train = load_clickbait_data('instances_train.jsonl', 'truth_train.jsonl')
val_headlines, y_val = load_clickbait_data('instances_validation.jsonl', 'truth_validation.jsonl')
test_headlines, y_test = load_clickbait_data('instances_test.jsonl', 'truth_test.jsonl')

print(f"\n✅ All Data Loaded Successfully!")
print(f"Train size: {len(train_headlines)}")
print(f"Validation size: {len(val_headlines)}")
print(f"Test size: {len(test_headlines)}\n")

# 3. Generate Semantic Embeddings
print("Loading Transformer model (all-MiniLM-L6-v2)...")
transformer = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding Training headlines (this will take a moment)...")
X_train = transformer.encode(train_headlines, show_progress_bar=True)

# For the final report benchmark, we test on the Test set
print("Encoding Testing headlines...")
X_test = transformer.encode(test_headlines, show_progress_bar=True)

# 4. Initialize Models
models = {
    "Support Vector Machine": SVC(kernel='linear', C=1.0, probability=True, random_state=42),
    "Multi-Layer Perceptron": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, activation='relu', random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
}

results = []

# 5. Train and Evaluate
print("\n--- Starting Model Training & Evaluation ---\n")

for name, model in models.items():
    print(f"Training {name}...")
    start_time = time.time()

    # Train on 2017 Train
    model.fit(X_train, y_train)

    # Predict on 2017 Test
    y_pred = model.predict(X_test)

    execution_time = time.time() - start_time

    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    results.append({
        "Model": name,
        "Test Accuracy": acc * 100,
        "Test F1-Score": f1 * 100,
        "Training Time (s)": execution_time
    })

    print(f"✅ {name} Complete!")
    print(classification_report(y_test, y_pred, target_names=['Non-Clickbait', 'Clickbait']))
    print("-" * 50 + "\n")

# 6. Output Final Benchmarking Table
print("=== FINAL BENCHMARK COMPARISON ===")
results_df = pd.DataFrame(results)
results_df['Test Accuracy'] = results_df['Test Accuracy'].round(2).astype(str) + '%'
results_df['Test F1-Score'] = results_df['Test F1-Score'].round(2).astype(str) + '%'
results_df['Training Time (s)'] = results_df['Training Time (s)'].round(3)

print(results_df.to_markdown(index=False))

Loading instances_train.jsonl and truth_train.jsonl...
Loading instances_validation.jsonl and truth_validation.jsonl...
Loading instances_test.jsonl and truth_test.jsonl...

✅ All Data Loaded Successfully!
Train size: 2459
Validation size: 19538
Test size: 18979

Loading Transformer model (all-MiniLM-L6-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Encoding Training headlines (this will take a moment)...


Batches:   0%|          | 0/77 [00:00<?, ?it/s]

Encoding Testing headlines...


Batches:   0%|          | 0/594 [00:00<?, ?it/s]


--- Starting Model Training & Evaluation ---

Training Support Vector Machine...
✅ Support Vector Machine Complete!
               precision    recall  f1-score   support

Non-Clickbait       0.85      0.87      0.86     14464
    Clickbait       0.56      0.52      0.54      4515

     accuracy                           0.79     18979
    macro avg       0.71      0.70      0.70     18979
 weighted avg       0.78      0.79      0.79     18979

--------------------------------------------------

Training Multi-Layer Perceptron...
✅ Multi-Layer Perceptron Complete!
               precision    recall  f1-score   support

Non-Clickbait       0.86      0.78      0.82     14464
    Clickbait       0.46      0.61      0.53      4515

     accuracy                           0.74     18979
    macro avg       0.66      0.69      0.67     18979
 weighted avg       0.77      0.74      0.75     18979

--------------------------------------------------

Training XGBoost...
✅ XGBoost Complete!
   

In [2]:
import pandas as pd
import numpy as np
import time
from sentence_transformers import SentenceTransformer
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. Helper function to load data
def load_clickbait_data(instances_file, truth_file):
    print(f"Loading {instances_file} and {truth_file}...")
    instances_df = pd.read_json(instances_file, lines=True)
    truth_df = pd.read_json(truth_file, lines=True)
    merged_df = pd.merge(instances_df, truth_df, on='id')

    if 'postText' in merged_df.columns:
        headlines = merged_df['postText'].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x)).tolist()
    elif 'targetTitle' in merged_df.columns:
        headlines = merged_df['targetTitle'].tolist()
    else:
        headlines = merged_df['headline'].tolist()

    if merged_df['truthClass'].dtype == object:
        labels = (merged_df['truthClass'] == 'clickbait').astype(int).tolist()
    else:
        labels = merged_df['truthClass'].tolist()
    return headlines, labels

# 2. Load All Data
train_headlines, y_train = load_clickbait_data('instances_train.jsonl', 'truth_train.jsonl')
val_headlines, y_val = load_clickbait_data('instances_validation.jsonl', 'truth_validation.jsonl')
test_headlines, y_test = load_clickbait_data('instances_test.jsonl', 'truth_test.jsonl')

# Calculate the exact imbalance ratio
imbalance_ratio = y_train.count(0) / y_train.count(1)
print(f"\n✅ Data Loaded. Class Imbalance Ratio: {imbalance_ratio:.2f} to 1")

# 3. Generate Embeddings (This takes a moment, but NO Grid Search!)
print("\nLoading Heavyweight Transformer (paraphrase-mpnet-base-v2)...")
transformer = SentenceTransformer('paraphrase-mpnet-base-v2')

X_train = transformer.encode(train_headlines, show_progress_bar=True)
X_val = transformer.encode(val_headlines, show_progress_bar=True)
X_test = transformer.encode(test_headlines, show_progress_bar=True)

# Combine Train and Val arrays so the models have maximum data to learn from
X_tune = np.vstack((X_train, X_val))
y_tune = np.concatenate((y_train, y_val))

# 4. The "Fast-Forward" Models (Hard-coded with the optimal Grid Search results)
print("\nInitializing Optimized Models...")
models = {
    # SVM optimized with RBF kernel and C=10
    "Support Vector Machine": SVC(C=10.0, kernel='rbf', probability=True, random_state=42, class_weight='balanced'),

    # MLP optimized with standard layers and low alpha
    "Multi-Layer Perceptron": MLPClassifier(hidden_layer_sizes=(128, 64), alpha=0.0001, max_iter=500, random_state=42),

    # XGBoost optimized with standard depth
    "XGBoost": XGBClassifier(max_depth=5, learning_rate=0.1, n_estimators=200, random_state=42, scale_pos_weight=imbalance_ratio, eval_metric='logloss')
}

results = []

# 5. Train and Evaluate (Single Pass - Lightning Fast)
print("\n--- Starting Optimized Model Training & Evaluation ---\n")

for name, model in models.items():
    print(f"Training {name}...")
    start_time = time.time()

    # Train directly on the combined tuning data (Train + Val)
    model.fit(X_tune, y_tune)

    # Predict strictly on the unseen Test data
    y_pred = model.predict(X_test)

    execution_time = time.time() - start_time

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    results.append({
        "Model": name,
        "Test Accuracy": acc * 100,
        "Test F1-Score": f1 * 100,
        "Total Time (s)": execution_time
    })

    print(f"✅ {name} Complete!")
    print(classification_report(y_test, y_pred, target_names=['Non-Clickbait', 'Clickbait']))
    print("-" * 50 + "\n")

# 6. Final Output
print("=== FINAL OPTIMIZED BENCHMARK COMPARISON ===")
results_df = pd.DataFrame(results)
results_df['Test Accuracy'] = results_df['Test Accuracy'].round(2).astype(str) + '%'
results_df['Test F1-Score'] = results_df['Test F1-Score'].round(2).astype(str) + '%'
results_df['Total Time (s)'] = results_df['Total Time (s)'].round(3)

print(results_df.to_markdown(index=False))

Loading instances_train.jsonl and truth_train.jsonl...
Loading instances_validation.jsonl and truth_validation.jsonl...
Loading instances_test.jsonl and truth_test.jsonl...

✅ Data Loaded. Class Imbalance Ratio: 2.23 to 1

Loading Heavyweight Transformer (paraphrase-mpnet-base-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/77 [00:00<?, ?it/s]

Batches:   0%|          | 0/611 [00:00<?, ?it/s]

Batches:   0%|          | 0/594 [00:00<?, ?it/s]


Initializing Optimized Models...

--- Starting Optimized Model Training & Evaluation ---

Training Support Vector Machine...
✅ Support Vector Machine Complete!
               precision    recall  f1-score   support

Non-Clickbait       0.89      0.91      0.90     14464
    Clickbait       0.69      0.63      0.66      4515

     accuracy                           0.85     18979
    macro avg       0.79      0.77      0.78     18979
 weighted avg       0.84      0.85      0.84     18979

--------------------------------------------------

Training Multi-Layer Perceptron...
✅ Multi-Layer Perceptron Complete!
               precision    recall  f1-score   support

Non-Clickbait       0.88      0.88      0.88     14464
    Clickbait       0.62      0.63      0.62      4515

     accuracy                           0.82     18979
    macro avg       0.75      0.75      0.75     18979
 weighted avg       0.82      0.82      0.82     18979

--------------------------------------------------
